# DeepSurv Neural Network Stratified by Batch


The notebook provides an interactive walk-through of the stratified **DeepSurv** neural network (NN) pipeline for survival prediction using simulated micro-RNA sequencing (miRNA-seq) data. 

The NN is built on top of the `pycox` framework, with addition implementation of stratification by "batches"--simulated data artifacts in the training/testing miRNA-seq data.

First, import necessary libraries and DeepSurv functions:

In [1]:
# basics
import numpy as np
import os
os.chdir("../..")

# deepsurv
from pss.utils import load_prepared_split
from pss.pycox.models import CoxPH, CoxPHStratified, StratifiedDataset
from pss.pycox.evaluation.eval_surv import EvalSurv
from pss.run_models import DeepSurvPipeline

### Load simulated data

* The input data are CVAE-augmented miRNA-seq expression counts for 538 markers.

* The outcomes are simulated survival time and censoring status, with batch ID supplied for stratification.

As an example, we will load the simulated miRNA-seq data and survival outcome from one of the experimental conditions.

In [ ]:
batchNormType = 'BE11Asso00_normNone'
dataType = 'linear-p10'
train_size = 1000
iter_i = 1

Specifically,

* `batchNormType`: batch effect (BE) presence in train/test data, batch-survival association, and data normalization method.

    * BE`{he_train}{he_test}`Asso`{sort_train}{sort_test}`: 1 indicates the presence of BE and/or non-zero correlation between BE and survival outcome in the dataset; 0 otherwise

    * norm`{norm_type}`: specifies one of the normalization methods--None (Raw), Total Count (TC), Upper Quartile (UQ), Median (median), Trimmed Mean of M-values (TMM), DESeq, and Quantile Normalization (QN).


* `dataType`: marker-survival association (`linear/nonlinera`) and number of true markers (`p=5, 10, 30`) 

    * The effect sizes (beta coefficients) for the true markers were chosen so that the total signal magnitude remains relatively consistent across varying true marker sizes.

    * There is an additional condition with 5 true markers but doubled the beta coefficients (`p=5-2x`)

* `train_size`: train set size N = 100, 200, 500, 1000, and 5000.

* `iter_i`: for each simulation condition, we run for 20 iterations with different train/test data.

<br>

For demonstration purposes, we choose:

* `batchNormType` = `BE11Asso00_normNone`

* `dataType` =  `linear-p10`

* `train_size` = 1000

* `iter_i` = 1



In [3]:
train_df, test_df = load_prepared_split(batchNormType=batchNormType,
                                        dataName=dataType,
                                        keep_batch=True,
                                        train_size=train_size,
                                        iter_i=iter_i)

print(f"Training data dimensions: {train_df.shape}")
print(f"Testing data dimensions:  {test_df.shape}")

Training data dimensions: (1000, 541)
Testing data dimensions:  (1000, 541)


### Create DeepSurv object

To allow for stratification, set `is_stratified=True`.

The DeepSurv pipeline also supports hyperparameter tuning via the `optuna` library. 

To enable tuning, specify the hyperparameter search grid and call the `.tune_hyperparameter()` function.

This will create a new study in a SQLite database (*.db) where all tuning trials will be automatically saved. Customize the database name through the `storage_url` argument.

In [4]:
hyperparameters = {
    "num_nodes": {"type": "categorical", "choices": [
        "128",
        "64",
        "32",
        "32-16",
        "64-32",
        "128-64",
        "64-64-32",
        "32-32-16"
        ]},
    "dropout": {"type": "float", "low": 0.1, "high": 0.5},
    "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
    "learning_rate": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True},
    "batch_size": {"type": "categorical", "choices": [128, 64, 32, 16]}
}
dl = DeepSurvPipeline(
    train_df=None, test_df=None,
    batchNormType=batchNormType,
    dataName=dataType,
    hyperparameters=hyperparameters,
    is_stratified=True,
    storage_url="sqlite:///deepsurv-torch-hp-test.db"
)

### Hperparameter tuning

Optuna tuning parameters:

* `n_splits`: number of folds in the cross-validation; default to 5.

* `n_trials`: number of tuning trials for the search; default to 30.

* `trial_threshold`: minimum number of existing completed trials in the study for the pipeline to skip tuning and directly extract the current best hyperparameters; default to 30.

* `n_jobs`: number of trials that can be run in parallel; default to 1.

In [ ]:
# Tune via optuna for automatic trials
n_splits=5
n_trials=30
trial_threshold=30
n_jobs=1

_ = dl.tune_hyperparameters(
    train_df,
    n_samples=train_size,
    n_splits=n_splits,
    n_trials=n_trials, 
    trial_threshold=trial_threshold,
    n_jobs=n_jobs
)

[I 2026-05-13 12:57:54,127] A new study created in RDB with name: BE11Asso00_normNone-linear-p10-stratified-deepsurv-torch-1000


⚠️No completed trials in Optuna study 'BE11Asso00_normNone-linear-p10-stratified-deepsurv-torch-1000'. Start hyperparameter tuning...


[I 2026-05-13 12:58:20,327] Trial 0 finished with value: 0.780188709388007 and parameters: {'num_nodes': '64', 'dropout': 0.3651208970997897, 'weight_decay': 0.00028487539871876447, 'learning_rate': 0.00012922454924988965, 'batch_size': 64}. Best is trial 0 with value: 0.780188709388007.
[I 2026-05-13 12:58:37,715] Trial 1 finished with value: 0.7877113193618969 and parameters: {'num_nodes': '128-64', 'dropout': 0.22934382991058275, 'weight_decay': 0.007837687063132745, 'learning_rate': 0.0009073813629975028, 'batch_size': 64}. Best is trial 1 with value: 0.7877113193618969.
[I 2026-05-13 12:59:40,423] Trial 2 finished with value: 0.7724172738072793 and parameters: {'num_nodes': '128-64', 'dropout': 0.18149112095710002, 'weight_decay': 0.00011490381894797433, 'learning_rate': 0.0014379036455112678, 'batch_size': 16}. Best is trial 1 with value: 0.7877113193618969.
[I 2026-05-13 13:00:19,517] Trial 3 finished with value: 0.7814910557465635 and parameters: {'num_nodes': '32', 'dropout': 

In [7]:
dl._best_params

{'num_nodes': '64',
 'dropout': 0.4755913542119763,
 'weight_decay': 0.0006580125160453848,
 'learning_rate': 0.0025695938809214215,
 'batch_size': 64}

### Understanding stratification

Stratification is achieved through a batch-stratified Cox partial likelihood loss.

The MLP network maps features to a log-risk score for each sample: 

$$log(h) = net(x)$$

which we use to compute the loss.

In the stratified NN case, this is used to compute a batch-specific loss within each batch/stratum. So for each batch:

```python
mask = (batch_indices == batch) # filter for only samples from this batch
idx = torch.argsort(durations[mask], descending=True) # sort from max to min survival time
events_batch = events[mask][idx] # obtain filtered and sorted event indicator  
log_h_batch = log_h[mask][idx]   # obtain filtered and sorted log-risk scores
losses[i] = cox_ph_loss_sorted(log_h_batch, events_batch, eps) # compute within-batch loss
```

Repeat for all batches. The final stratified loss is the sum of batch-specific losses:

```python
return losses.sum()
```

### Training the neural network with stratified loss

The `train()` function will return:
- runtime  
- train brier 
- test brier  
- train c-index  
- test c-index

In [8]:
dl.train(train_df=train_df, test_df=test_df)


(6.34,
 0.07579440570003036,
 0.10022344359411711,
 0.8402829885303891,
 0.7740424791086351)

## Breakdown of steps in `train()`

In [9]:
import torch
import torchtuples as tt
from sklearn.preprocessing import StandardScaler
from sklearn_pandas import DataFrameMapper

time_col = "time"
status_col = "status"
batch_col = "batch_id"

In [10]:
def _preprocess_data(df, mapper=None, fit_scaler=True):
    survival_cols = [time_col, status_col]
    covariate_cols = [col for col in df.columns if col not in survival_cols]
    # Transform features (miRNA expression)
    if fit_scaler or mapper is None:
        standardize = [([col], StandardScaler()) for col in covariate_cols]
        mapper = DataFrameMapper(standardize)
        x = mapper.fit_transform(df[covariate_cols]).astype('float32')
    else:
        x = mapper.transform(df[covariate_cols]).astype('float32')
    # Prepare labels (survival data)
    y = (df[time_col].values, df[status_col].values)
    
    return x, y, mapper

batch_ids_train = train_df[batch_col].to_numpy().reshape(-1)
batch_ids_test = test_df[[batch_col]].to_numpy().reshape(-1)

train_sub = train_df.drop(columns=[batch_col])
test_sub = test_df.drop(columns=[batch_col])

x_train, y_train, mapper = _preprocess_data(train_sub)
x_test, y_test, _ = _preprocess_data(test_sub, mapper=mapper, fit_scaler=False)

durations_train, events_train = y_train[0], y_train[1]
durations_test, events_test = y_test[0], y_test[1]

# Prepare data 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_train = torch.from_numpy(x_train).to(device)
x_test = torch.from_numpy(x_test).to(device)

batch_ids_train = torch.from_numpy(batch_ids_train).long().to(device)
batch_ids_test   = torch.from_numpy(batch_ids_test).long().to(device)
durations_train = torch.from_numpy(durations_train).float().to(device)
durations_test = torch.from_numpy(durations_test).float().to(device)
events_train = torch.from_numpy(events_train).float().to(device)
events_test = torch.from_numpy(events_test).float().to(device)
y_train = (durations_train, events_train)
y_test = (durations_test, events_test)

print(device)

cpu


In [11]:
def _parse_num_nodes(s):
    return [int(x) for x in s.split("-")]

params = dl._best_params
input_size = x_train.shape[1]
output_size = 1
num_nodes = params.get("num_nodes", "32-16")            # Default num of layers & nodes
num_nodes = _parse_num_nodes(num_nodes)
dropout = params.get("dropout", 0.1)                    # Default dropout rate
learning_rate = params.get("learning_rate", 1e-3)       # Default learning rate
batch_size = params.get("batch_size", 128)              # Default batch size
epochs = params.get("epochs", 500)                      # Default number of epochs
batch_norm = params.get("batch_norm", True)             # Default batch normalization
output_bias = params.get("output_bias", True)           # Default output bias
weight_decay = params.get("weight_decay", 1e-4)         # Default weight decay
activation_map = {                                      
    "ReLU": torch.nn.ReLU,
    "LeakyReLU": torch.nn.LeakyReLU,
    "SELU": torch.nn.SELU
} 
activation = activation_map.get(params.get("activation", "ReLU")) # Activation function

In [14]:
net = tt.practical.MLPVanilla(
    in_features=input_size,
    out_features=output_size,
    num_nodes=num_nodes,
    dropout=dropout, 
    batch_norm=batch_norm,
    activation=activation,
    output_bias=output_bias
).to(device)
optimizer = tt.optim.Adam(weight_decay=weight_decay, lr=learning_rate)

# Get default early stopping settings if not defined 
patience = 30
min_delta = 1e-3
callbacks = [tt.callbacks.EarlyStopping(patience=patience, min_delta=min_delta)]

### Stratified Cox NN

In [12]:
train_dataset = StratifiedDataset(x_train, durations_train, events_train, batch_ids_train)
test_dataset = StratifiedDataset(x_test, durations_test, events_test, batch_ids_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

for xb, db, eb, bb in train_loader:
    print("Train batch total events:", int(eb.sum().item()),
          "| per-stratum:", {int(s): int(eb[bb==s].sum().item()) for s in bb.unique().tolist()})

Train batch total events: 39 | per-stratum: {1: 1, 2: 1, 3: 2, 4: 1, 5: 3, 6: 4, 7: 3, 8: 5, 9: 2, 10: 3, 11: 4, 12: 3, 13: 2, 14: 3, 15: 2}
Train batch total events: 47 | per-stratum: {1: 2, 2: 3, 3: 3, 4: 2, 5: 3, 6: 4, 7: 5, 8: 3, 9: 2, 10: 4, 11: 4, 12: 2, 13: 3, 14: 2, 15: 5}
Train batch total events: 49 | per-stratum: {1: 3, 2: 2, 3: 5, 4: 5, 5: 2, 6: 3, 7: 3, 8: 3, 9: 4, 10: 4, 11: 3, 12: 1, 13: 3, 14: 4, 15: 4}
Train batch total events: 46 | per-stratum: {1: 2, 2: 3, 3: 2, 4: 3, 5: 3, 6: 3, 7: 2, 8: 3, 9: 5, 10: 3, 11: 5, 12: 3, 13: 1, 14: 5, 15: 3}
Train batch total events: 54 | per-stratum: {1: 5, 2: 4, 3: 5, 4: 7, 5: 0, 6: 2, 7: 5, 8: 5, 9: 4, 10: 4, 11: 3, 12: 6, 13: 1, 14: 2, 15: 1}
Train batch total events: 44 | per-stratum: {1: 1, 2: 4, 3: 4, 4: 4, 5: 5, 6: 3, 7: 4, 8: 6, 9: 2, 10: 4, 11: 2, 12: 2, 13: 1, 14: 0, 15: 2}
Train batch total events: 46 | per-stratum: {1: 4, 2: 3, 3: 0, 4: 5, 5: 3, 6: 3, 7: 2, 8: 1, 9: 4, 10: 3, 11: 2, 12: 4, 13: 4, 14: 4, 15: 4}
Train batch t

In [15]:
import time

# Stratified CoxPH model
model = CoxPHStratified(net, optimizer=optimizer)
model.metrics = {'val_loss': model.loss}
start = time.time() # Record iteration start time
log = model.fit_dataloader(
    train_loader,
    epochs=epochs,
    callbacks=callbacks,
    verbose=True,
    val_dataloader=test_loader  # optional for now
)
stop = time.time() # Record time when training finished
duration = round(stop - start, 2)
print(f"Training time: {duration}")

0:	[0s / 0s],		train_loss: 12.8731,	val_loss: 10.4956
1:	[0s / 0s],		train_loss: 11.0012,	val_loss: 9.7608
2:	[0s / 0s],		train_loss: 10.3254,	val_loss: 9.4684
3:	[0s / 0s],		train_loss: 9.4226,	val_loss: 9.3564
4:	[0s / 0s],		train_loss: 9.3654,	val_loss: 9.3314
5:	[0s / 0s],		train_loss: 10.0344,	val_loss: 9.4816
6:	[0s / 0s],		train_loss: 8.8521,	val_loss: 9.4887
7:	[0s / 0s],		train_loss: 9.2038,	val_loss: 9.1326
8:	[0s / 0s],		train_loss: 8.9925,	val_loss: 8.9477
9:	[0s / 0s],		train_loss: 9.1812,	val_loss: 8.8757
10:	[0s / 0s],		train_loss: 8.9136,	val_loss: 9.2315
11:	[0s / 1s],		train_loss: 9.0770,	val_loss: 8.6986
12:	[0s / 1s],		train_loss: 8.6194,	val_loss: 9.0961
13:	[0s / 1s],		train_loss: 9.0551,	val_loss: 8.8808
14:	[0s / 1s],		train_loss: 8.4779,	val_loss: 9.0933
15:	[0s / 1s],		train_loss: 9.1779,	val_loss: 9.4429
16:	[0s / 1s],		train_loss: 8.7341,	val_loss: 8.9302
17:	[0s / 1s],		train_loss: 8.5969,	val_loss: 8.8924
18:	[0s / 1s],		train_loss: 8.8263,	val_loss: 8.860

#### Evaluation: *Stratified C-index*

In [ ]:
# ==================== Evaluation ====================
# Convert torch tensors back to numpy objects for evaluation
durations_train_np = durations_train.detach().cpu().numpy()
durations_test_np  = durations_test.detach().cpu().numpy()
events_train_np    = events_train.detach().cpu().numpy()
events_test_np     = events_test.detach().cpu().numpy()
batch_ids_train_np = batch_ids_train.detach().cpu().numpy()
batch_ids_test_np   = batch_ids_test.detach().cpu().numpy()

# Compute baseline hazards (per-batch)
_ = model.compute_baseline_hazards(
    input=x_train, 
    target=(durations_train, events_train), 
    batch_ids=batch_ids_train_np
    )

# Initialize EvalSurv objects 
tr_surv  = model.predict_surv_df(x_train, batch_ids = batch_ids_train_np)
te_surv = model.predict_surv_df(x_test, batch_ids = batch_ids_test_np)
tr_ev = EvalSurv(tr_surv, durations_train_np, events_train_np, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test_np, events_test_np, censor_surv='km')

# Concordance index ----------------
tr_strat_c_index  = tr_ev.stratified_concordance_td(batch_indices=batch_ids_train_np) 
te_strat_c_index = te_ev.stratified_concordance_td(batch_indices=batch_ids_test_np) 

print("Train:", tr_strat_c_index, "  |  Test:", te_strat_c_index)

Train:  0.8395326401543574   |  Test: 0.7827123955431755


In [17]:
# Manual test
from pss.pycox.evaluation.concordance import concordance_td

batch_indices = batch_ids_test.detach().cpu().numpy() if not isinstance(batch_ids_test, np.ndarray) else batch_ids_train
batches = np.unique(batch_indices)
c_index_ls , n_pairs_ls = np.zeros(len(batches)), np.zeros(len(batches))

for i, batch in enumerate(batches):
    # Filter data by batch
    mask = (batch_indices == batch)
    if mask.sum() == 0:
        continue  # skip empty batch
    batch_durations = durations_test_np[mask]
    batch_events = events_test_np[mask]
    batch_surv = te_ev.surv.iloc[:, mask]
    if batch_events.sum() == 0:
        continue
    
    # Compute concordance for the current batch
    c_index_batch, n_pairs_batch = concordance_td(
        batch_durations, batch_events, batch_surv.values,
        te_ev.idx_at_times(batch_durations), method='adj_antolini'
    )
    n_pairs_ls[i] = n_pairs_batch
    c_index_ls[i] = c_index_batch
    
print("Final score: %f" % (np.sum(c_index_ls*n_pairs_ls) / np.sum(n_pairs_ls) if np.sum(n_pairs_ls) > 0 else float('nan')))

for e, c in zip(n_pairs_ls, c_index_ls):
    print(f"{int(e)} comparable pairs: {round(c,3)}")

Final score: 0.782712
1909 comparable pairs: 0.754
1796 comparable pairs: 0.798
1933 comparable pairs: 0.765
1746 comparable pairs: 0.833
1951 comparable pairs: 0.744
1973 comparable pairs: 0.795
2037 comparable pairs: 0.746
2040 comparable pairs: 0.834
2052 comparable pairs: 0.851
2006 comparable pairs: 0.728
1941 comparable pairs: 0.759
1659 comparable pairs: 0.824
1933 comparable pairs: 0.761
1840 comparable pairs: 0.815
1904 comparable pairs: 0.744


#### *One-batch C-index* 

In [19]:
baseline_hazards_1batch = model.compute_baseline_hazards(input=x_train, target=(durations_train, events_train))

# Initialize EvalSurv objects 
tr_surv = model.predict_surv_df(x_train, baseline_hazards_= baseline_hazards_1batch)
te_surv = model.predict_surv_df(x_test, baseline_hazards_ = baseline_hazards_1batch)
tr_ev = EvalSurv(tr_surv, durations_train_np, events_train_np, censor_surv = 'km')
te_ev = EvalSurv(te_surv, durations_test_np, events_test_np, censor_surv = 'km')

# Concordance index (non-stratified) ----------------
tr_c_index, _  = tr_ev.concordance_td() 
te_c_index, _ = te_ev.concordance_td() 
print("Train:", tr_c_index, "  |  Test:", te_c_index)

Train: 0.8227056652048509   |  Test: 0.7904149062741421


### Non-stratified Cox NN

In [ ]:
net = tt.practical.MLPVanilla(
    in_features=input_size,
    out_features=output_size,
    num_nodes=num_nodes,
    dropout=dropout, 
    batch_norm=batch_norm,
    activation=activation,
    output_bias=output_bias
).to(device)
optimizer = tt.optim.Adam(weight_decay=weight_decay, lr=learning_rate)


# Get default early stopping settings if not defined 
patience = 30
min_delta = 1e-3
callbacks = [tt.callbacks.EarlyStopping(patience=patience, min_delta=min_delta)]

In [24]:
# CoxPH model
model_nonstrat = CoxPH(net, optimizer=optimizer)
log = model_nonstrat.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks, 
    verbose=True,
    val_data=(x_test, y_test),
    val_batch_size=batch_size
)

0:	[0s / 0s],		train_loss: 3.5175,	val_loss: 3.1834
1:	[0s / 0s],		train_loss: 3.2676,	val_loss: 3.1253
2:	[0s / 0s],		train_loss: 3.1519,	val_loss: 3.0597
3:	[0s / 0s],		train_loss: 3.0711,	val_loss: 2.9916
4:	[0s / 0s],		train_loss: 2.9686,	val_loss: 2.9683
5:	[0s / 0s],		train_loss: 2.9730,	val_loss: 2.9440
6:	[0s / 0s],		train_loss: 2.9922,	val_loss: 2.9545
7:	[0s / 0s],		train_loss: 2.9659,	val_loss: 2.9365
8:	[0s / 0s],		train_loss: 2.9326,	val_loss: 2.9298
9:	[0s / 0s],		train_loss: 2.8984,	val_loss: 2.8936
10:	[0s / 0s],		train_loss: 2.8917,	val_loss: 2.8891
11:	[0s / 0s],		train_loss: 2.9479,	val_loss: 2.8845
12:	[0s / 0s],		train_loss: 2.8842,	val_loss: 2.8841
13:	[0s / 0s],		train_loss: 2.9059,	val_loss: 2.8826
14:	[0s / 0s],		train_loss: 2.8529,	val_loss: 2.8841
15:	[0s / 0s],		train_loss: 2.9167,	val_loss: 2.8940
16:	[0s / 0s],		train_loss: 2.8779,	val_loss: 2.8891
17:	[0s / 0s],		train_loss: 2.8437,	val_loss: 2.8998
18:	[0s / 0s],		train_loss: 2.8626,	val_loss: 2.9082
19:

In [26]:
# ==================== Evaluation ====================
_ = model_nonstrat.compute_baseline_hazards(
    input=x_train, 
    target=(durations_train, events_train)
    )

# Initialize EvalSurv objects 
tr_surv  = model_nonstrat.predict_surv_df(x_train)
te_surv = model_nonstrat.predict_surv_df(x_test)
tr_ev = EvalSurv(tr_surv, durations_train_np, events_train_np, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test_np, events_test_np, censor_surv='km')

# Concordance index ----------------
orig_tr_c_index, orig_tr_n_paris  = tr_ev.concordance_td() 
orig_te_c_index, orig_te_n_pairs = te_ev.concordance_td() 

print("Train", orig_tr_c_index, "  |  Test:", orig_te_c_index)

Train 0.836161765886035   |  Test: 0.8007537498366646


In [27]:
# Concordance index ----------------
orig_tr_strat_c_index  = tr_ev.stratified_concordance_td(batch_indices=batch_ids_train_np) 
orig_te_strat_c_index = te_ev.stratified_concordance_td(batch_indices=batch_ids_test_np) 

print("Train", orig_tr_strat_c_index, "  |  Test:", orig_te_strat_c_index)

Train 0.8353878586486583   |  Test: 0.8012883008356546


### Results comparison

In [ ]:
results = {}
results["stratified_nn_strat_cindex"] =  [tr_strat_c_index, te_strat_c_index]
results["stratified_nn_non_strat_cindex"] = [tr_c_index, te_c_index]
results["original_nn_non_strat_cindex"] = [orig_tr_c_index, orig_te_c_index]
results["original_nn_strat_cindex"] = [orig_tr_strat_c_index, orig_te_strat_c_index]

import pandas as pd
pd.DataFrame(results).rename()

,stratified_nn_strat_cindex,stratified_nn_non_strat_cindex,original_nn_non_strat_cindex,original_nn_strat_cindex
0,0.839533,0.822706,0.836162,0.835388
1,0.782712,0.790415,0.800754,0.801288
